# Quant Finance – Learning Notebook
# Module 2: Risk Measures, Portfolio Theory & CAPM

**Author:** Florian Ebner

## Source References
- Sharpe, W.F. (1966). *Mutual Fund Performance*. Journal of Business, 39(1).
- Sortino, F. & van der Meer, R. (1991). *Downside Risk*. Journal of Portfolio Management.
- Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*. McGraw-Hill.
- Jorion, P. (2007). *Value at Risk* (3rd ed.). McGraw-Hill.
- Basel Committee on Banking Supervision (2019). *Minimum Capital Requirements for Market Risk (FRTB)*.
- Markowitz, H. (1952). *Portfolio Selection*. Journal of Finance.
- Sharpe, W.F. (1964). *Capital Asset Prices*. Journal of Finance.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

np.random.seed(42)
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.color': '#e0e0e0',
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})
TRADING_DAYS = 252
RISK_FREE    = 0.035
rf_d         = RISK_FREE / TRADING_DAYS
T            = 756
print('Module 2 – Risk Measures, Portfolio Theory & CAPM')

---
## 9. Sharpe Ratio

### Theory

The **Sharpe Ratio** measures **risk-adjusted return** per unit of total risk (volatility):

$$S = \frac{\mu_p - r_f}{\sigma_p}$$

- $\mu_p$: Annualised portfolio return
- $r_f$: Risk-free rate (e.g. 3-month Euribor, currently ~3.5%)
- $\sigma_p$: Annualised portfolio volatility

**Interpretation:** How much excess return above the risk-free rate do I earn per unit of risk taken?
- Sharpe < 0: Worse than risk-free parking
- Sharpe 0–0.5: Weak
- Sharpe 0.5–1.0: Good (typical for a passive market index)
- Sharpe > 1.0: Very good (targeted by hedge funds)
- Sharpe > 2.0: Excellent (rare; often an overfitting warning signal)

### Critiques and Limitations

1. **Symmetric penalisation:** Sharpe penalises upside and downside volatility equally. Large positive outliers reduce the Sharpe Ratio — which is intuitively wrong.
2. **Normality assumption:** Sharpe is optimal only under normality. With fat tails or skewness it underestimates true risk.
3. **Time dependence:** Sharpe scales with $\sqrt{T}$ — a strategy with a daily Sharpe of 0.05 has an annualised Sharpe of 0.05 × √252 ≈ 0.79. This is sometimes exploited misleadingly.

**Source:** Sharpe (1966, 1994); Lo, A. (2002). *The Statistics of Sharpe Ratios*. Financial Analysts Journal

In [ ]:
strats = {
    'Strategy A (High Return, High Vol)':   (0.15, 0.25),
    'Strategy B (Moderate, Low Vol)':        (0.09, 0.12),
    'Strategy C (Low Return, Very Low Vol)': (0.055, 0.06),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']

print(f'{"Strategy":<42} {"Return":>8} {"Vol":>8} {"Sharpe":>8}')
print('-' * 68)

for (name, (mu, sig)), col in zip(strats.items(), colors):
    r    = np.random.normal(mu/TRADING_DAYS, sig/np.sqrt(TRADING_DAYS), T)
    cum  = (1 + r).cumprod()
    cagr = cum[-1] ** (TRADING_DAYS/T) - 1
    rvol = r.std() * np.sqrt(TRADING_DAYS)
    sr   = (cagr - RISK_FREE) / rvol
    print(f'{name:<42} {cagr*100:>7.2f}% {rvol*100:>7.2f}% {sr:>8.3f}')
    axes[0].plot(cum, color=col, lw=2, label=f'{name.split("(")[0].strip()} (SR={sr:.2f})')

axes[0].set_title('Cumulative Value – 3 Strategies', fontweight='bold')
axes[0].set_ylabel('Cumulative Value')
axes[0].legend(fontsize=9)

vols_range = np.linspace(0.05, 0.40, 200)
fixed_ret  = 0.10
sharpes    = (fixed_ret - RISK_FREE) / vols_range
axes[1].plot(vols_range * 100, sharpes, color='#4a90d9', lw=2)
axes[1].axhline(1.0, color='green',  ls='--', lw=1, label='Sharpe = 1 (very good)')
axes[1].axhline(0.5, color='orange', ls='--', lw=1, label='Sharpe = 0.5 (good)')
axes[1].set_xlabel('Annualised Volatility (%)')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_title(f'Sharpe Ratio vs. Volatility (Return={fixed_ret*100:.0f}%, rf={RISK_FREE*100:.1f}%)',
                  fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Sortino Ratio

### Theory

The **Sortino Ratio** fixes the biggest criticism of the Sharpe Ratio: it penalises only **downside volatility**, not upside:

$$\text{Sortino} = \frac{\mu_p - r_f}{\sigma_{\text{downside}}}$$

**Downside Deviation** (also: semi-standard deviation below a target return $\tau$, typically $\tau = r_f$ or $\tau = 0$):

$$\sigma_{\text{downside}} = \sqrt{\frac{1}{T} \sum_{t: r_t < \tau} (r_t - \tau)^2}$$

### When to Prefer Sortino

- For strategies with **positive skewness** (many small gains, rare large gains), Sortino > Sharpe — correctly, because high upside variance is not a real risk.
- For **option strategies** (e.g. covered calls): Sharpe underestimates quality because gains are capped. Sortino is better.
- Pension funds with **minimum return obligations** set $\tau$ = liability return.

**Source:** Sortino & van der Meer (1991); Rollinger, T. & Hoffman, S. (2013). *Sortino: A Sharper Ratio*. CME Group

In [ ]:
def sortino_ratio(returns, target=None, rf=RISK_FREE):
    if target is None:
        target = rf / TRADING_DAYS
    ann_ret      = returns.mean() * TRADING_DAYS
    downside     = returns[returns < target] - target
    downside_std = np.sqrt((downside ** 2).mean()) * np.sqrt(TRADING_DAYS)
    return (ann_ret - rf) / downside_std if downside_std > 0 else np.inf

r_sym = pd.Series(np.random.normal(0.0004, 0.010, T))
r_pos = pd.Series(np.random.exponential(0.0008, T) - 0.0004
                  + np.random.normal(0, 0.006, T))

for name, r in [('Symmetric (Normal)', r_sym), ('Positively Skewed', r_pos)]:
    ann_r = r.mean() * TRADING_DAYS
    ann_v = r.std()  * np.sqrt(TRADING_DAYS)
    sh    = (ann_r - RISK_FREE) / ann_v
    so    = sortino_ratio(r)
    sk    = r.skew()
    print(f'{name}:')
    print(f'  Return={ann_r*100:.2f}%, Vol={ann_v*100:.2f}%, Skew={sk:.2f}')
    print(f'  Sharpe={sh:.3f}, Sortino={so:.3f}  (Sortino/Sharpe={so/sh:.2f}x)\n')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (name, r), col in zip([('Symmetric', r_sym), ('Pos. Skewed', r_pos)],
                           ['#4a90d9', '#2ecc71']):
    axes[0].hist(r * 100, bins=50, alpha=0.5, color=col, edgecolor='white',
                 label=name, density=True)
axes[0].axvline(rf_d * 100, color='red', ls='--', lw=1.5, label='Target = rf/252')
axes[0].set_title('Return Distribution: Symmetric vs. Skewed', fontweight='bold')
axes[0].set_xlabel('Daily Return (%)')
axes[0].legend()

metrics   = ['Sharpe', 'Sortino']
sym_vals  = [(r_sym.mean()*TRADING_DAYS - RISK_FREE)/(r_sym.std()*np.sqrt(TRADING_DAYS)),
             sortino_ratio(r_sym)]
skew_vals = [(r_pos.mean()*TRADING_DAYS - RISK_FREE)/(r_pos.std()*np.sqrt(TRADING_DAYS)),
             sortino_ratio(r_pos)]
x = np.arange(2)
axes[1].bar(x - 0.2, sym_vals,  0.35, label='Symmetric',   color='#4a90d9', edgecolor='white')
axes[1].bar(x + 0.2, skew_vals, 0.35, label='Pos. Skewed', color='#2ecc71', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
axes[1].set_title('Sharpe vs. Sortino Comparison', fontweight='bold')
axes[1].set_ylabel('Ratio')
axes[1].legend()
plt.tight_layout()
plt.savefig('sortino.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Information Ratio (IR)

### Theory

The **Information Ratio** measures an active manager's ability to **consistently generate alpha relative to a benchmark**:

$$IR = \frac{\alpha}{\sigma_{\text{Active}}} = \frac{\overline{r_p - r_b}}{\sigma(r_p - r_b)}$$

- $\alpha$: Average **active return** (portfolio minus benchmark)
- $\sigma_{\text{Active}}$: **Tracking Error** — volatility of the active return

### Tracking Error

**Tracking Error (TE)** = standard deviation of the return differential (portfolio - benchmark). Measures how far the manager deviates from the benchmark:
- Low TE (< 1%): Quasi-index fund
- Medium TE (2–5%): Typical active management
- High TE (> 8%): Very active or concentrated portfolio

### The Fundamental Law of Active Management (Grinold)

$$IR \approx IC \cdot \sqrt{BR}$$

- **IC (Information Coefficient):** Correlation between forecasts and actual returns (typically 0.05–0.10)
- **BR (Breadth):** Number of independent investment decisions per year

Implication: a manager with modest skill (IC=0.05) but 100 independent trades per year achieves IR ≈ 0.5 — better than a manager with IC=0.1 but only 10 trades per year (IR ≈ 0.32). **Diversification of decisions is as important as skill.**

**Source:** Grinold & Kahn (1999), Ch. 6 & 7

In [ ]:
r_benchmark   = pd.Series(np.random.normal(0.0003, 0.010, T))
active_return = pd.Series(np.random.normal(0.00012, 0.004, T))
r_portfolio   = r_benchmark + active_return

active_ret_ann = active_return.mean() * TRADING_DAYS
tracking_error = active_return.std()  * np.sqrt(TRADING_DAYS)
ir             = active_ret_ann / tracking_error

cum_bench = (1 + r_benchmark).cumprod()
cum_port  = (1 + r_portfolio).cumprod()

print(f'Active Return (ann.)  : {active_ret_ann*100:.3f}%')
print(f'Tracking Error (ann.) : {tracking_error*100:.3f}%')
print(f'Information Ratio     : {ir:.4f}')
print(f'\nFundamental Law (IC=0.05, BR=100): IR ≈ {0.05*np.sqrt(100):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cum_bench.values, color='#95a5a6', lw=2, label='Benchmark')
axes[0].plot(cum_port.values,  color='#2ecc71', lw=2, label=f'Portfolio (IR={ir:.2f})')
axes[0].set_title('Portfolio vs. Benchmark', fontweight='bold')
axes[0].set_ylabel('Cumulative Value')
axes[0].legend()

roll_ir = pd.Series([
    (active_return.iloc[max(0,i-63):i].mean() * TRADING_DAYS) /
    (active_return.iloc[max(0,i-63):i].std() * np.sqrt(TRADING_DAYS))
    if active_return.iloc[max(0,i-63):i].std() > 0 else 0
    for i in range(63, T)
])
axes[1].plot(roll_ir.values, color='#3498db', lw=1.5)
axes[1].axhline(ir,  color='green',  ls='--', lw=1.5, label=f'Full IR={ir:.2f}')
axes[1].axhline(0,   color='black',  lw=0.8)
axes[1].axhline(0.5, color='orange', ls=':',  lw=1, label='IR=0.5 (good)')
axes[1].set_title('Rolling Information Ratio (63-day)', fontweight='bold')
axes[1].set_ylabel('Information Ratio')
axes[1].legend()
plt.tight_layout()
plt.savefig('information_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Capital Asset Pricing Model (CAPM)

### Theory

The **CAPM** (Sharpe, 1964; Lintner, 1965) is the fundamental equilibrium model for security prices. It states: in equilibrium, only **systematic risk** (market risk, beta) is rewarded — **idiosyncratic risk** (firm-specific) is not rewarded because it can be eliminated through diversification.

**CAPM equation:**
$$E[r_i] = r_f + \beta_i \cdot (E[r_m] - r_f)$$

- $E[r_m] - r_f$: **Equity Risk Premium (ERP)** — the excess return of the market portfolio over the risk-free rate. Historically ~5–7% p.a. for US equities, ~4–5% for Europe.
- $\beta_i$: Sensitivity of the asset to the market

### Security Market Line (SML)

The **SML** is the graphical representation of CAPM: a line in beta/expected-return space. Assets *above* the SML are undervalued (positive alpha), assets *below* are overvalued.

### Critique of CAPM

1. **Empirically poor:** Many studies show that beta alone explains returns poorly — Fama-French and other factors deliver better explanatory power.
2. **Unobservable market portfolio:** The theoretical market portfolio contains *all* risky assets (equities, bonds, real estate, human capital). In practice an equity index is used as proxy.
3. **Static assumptions:** CAPM assumes constant expected returns and volatilities.

Despite this, CAPM remains the conceptual foundation for **cost of capital** (DCF models), **performance attribution**, and **risk-adjusted return** calculations.

**Source:** Sharpe (1964); Fama & French (2004). *The Capital Asset Pricing Model: Theory and Evidence*. Journal of Economic Perspectives

In [ ]:
r_market_ann = 0.08
erp          = r_market_ann - RISK_FREE

betas_true   = np.array([0.3, 0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0])
capm_returns = RISK_FREE + betas_true * erp
alphas_true  = np.random.normal(0, 0.015, len(betas_true))
obs_returns  = capm_returns + alphas_true

beta_range = np.linspace(0, 2.2, 200)
sml        = RISK_FREE + beta_range * erp

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(beta_range, sml * 100, 'b-', lw=2,
        label=f'SML: E[r] = {RISK_FREE*100:.1f}% + β × {erp*100:.1f}%')
ax.axhline(RISK_FREE * 100, color='grey', ls=':', lw=1,
           label=f'Risk-Free = {RISK_FREE*100:.1f}%')
ax.axvline(1.0, color='grey', ls=':', lw=1, label='β = 1 (Market Portfolio)')

for i, (b, r_obs, alpha) in enumerate(zip(betas_true, obs_returns, alphas_true)):
    r_sml = RISK_FREE + b * erp
    col   = '#2ecc71' if alpha > 0 else '#e74c3c'
    ax.scatter(b, r_obs * 100, color=col, s=80, zorder=5)
    ax.plot([b, b], [r_sml * 100, r_obs * 100], color=col, lw=1, ls='--', alpha=0.7)
    ax.annotate(f'A{i+1}\n(α={alpha*100:.1f}%)', (b, r_obs * 100),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

from matplotlib.patches import Patch
handles, labels = ax.get_legend_handles_labels()
handles += [Patch(color='#2ecc71', label='Positive Alpha (undervalued)'),
            Patch(color='#e74c3c', label='Negative Alpha (overvalued)')]
ax.legend(handles=handles, fontsize=9, loc='upper left')
ax.set_xlabel('Beta (β)', fontsize=12)
ax.set_ylabel('Expected Return (% p.a.)', fontsize=12)
ax.set_title('CAPM – Security Market Line (SML)', fontsize=14, fontweight='bold')
ax.scatter(1.0, r_market_ann * 100, color='#3498db', s=150, zorder=6, marker='*')
plt.tight_layout()
plt.savefig('capm_sml.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. VaR, CVaR and Expected Shortfall (Deep Dive)

### Theory

Covered extensively in Project 1. Here we focus on **mathematical properties** and the **regulatory evolution**.

### Coherent Risk Measures

Artzner et al. (1999) defined four axioms for **coherent risk measures**:
1. **Monotonicity:** If portfolio A always dominates B, its risk is lower
2. **Subadditivity:** Risk(A+B) ≤ Risk(A) + Risk(B) — diversification reduces risk
3. **Positive Homogeneity:** Double the position → double the risk
4. **Translation Invariance:** Add a risk-free asset → risk decreases accordingly

**VaR is not coherent** — it violates subadditivity. It is possible for two portfolios combined to have higher VaR than separated, which creates incentives against diversification.

**CVaR/ES is coherent** — it satisfies all four axioms. This is the primary mathematical reason Basel III switched to ES.

**Source:** Artzner, P. et al. (1999). *Coherent Measures of Risk*. Mathematical Finance, 9(3); Basel Committee (2019)

In [ ]:
port_vol = 0.18
port_ret = 0.08
pf_val   = 1_000_000

daily_r = np.random.normal(port_ret/TRADING_DAYS, port_vol/np.sqrt(TRADING_DAYS), 10000)
pnl     = daily_r * pf_val

var_hist_99 = -np.percentile(pnl, 1.0)
es_hist_975 = -pnl[pnl <= np.percentile(pnl, 2.5)].mean()

mu_pnl  = pnl.mean()
sig_pnl = pnl.std()
z_99    = stats.norm.ppf(0.01)
var_param = -(mu_pnl + z_99 * sig_pnl)
es_param  = -(mu_pnl - sig_pnl * stats.norm.pdf(z_99) / 0.025)

print('VaR and CVaR Method Comparison (99% VaR / 97.5% ES):')
print(f'  Historical Simulation : VaR={var_hist_99:,.0f}€, ES={es_hist_975:,.0f}€')
print(f'  Parametric (Normal)   : VaR={var_param:,.0f}€, ES={es_param:,.0f}€')

fig, ax = plt.subplots(figsize=(12, 5))
counts, bins, patches = ax.hist(pnl, bins=100, density=True, color='#4a90d9',
                                alpha=0.7, edgecolor='white', label='P&L Distribution')
ax.axvline(-var_hist_99, color='#e67e22', lw=2, ls='--',
           label=f'VaR 99% = €{var_hist_99:,.0f}')
ax.axvline(-es_hist_975, color='#c0392b', lw=2.5, ls='-',
           label=f'CVaR 97.5% = €{es_hist_975:,.0f}')
threshold = np.percentile(pnl, 2.5)
for patch, left in zip(patches, bins):
    if left < threshold:
        patch.set_facecolor('#d9534f'); patch.set_alpha(0.85)
ax.set_xlabel('P&L (€)')
ax.set_title('P&L Distribution: VaR vs. CVaR/ES  (ES = Avg. of red area)', fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
plt.tight_layout()
plt.savefig('var_cvar_deep.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 14. Modern Portfolio Theory – Estimation Error Problem

### Theory

Covered in Project 2. Here we focus on the **estimation error problem** and robust alternatives.

### The Estimation Error Problem (Michaud, 1989)

Markowitz optimisation **amplifies** estimation errors. Small errors in expected returns lead to drastically different optimal portfolios. The classic problem: the optimiser finds the portfolio with the *highest error in the return forecast*, not the truly best portfolio.

**Solutions:**
1. **Resampled Efficiency (Michaud):** Bootstrap the parameter estimation and average over many frontiers.
2. **Black-Litterman Model (1990):** Combine CAPM market equilibrium with subjective manager views. Starting point is not return forecasts but the market portfolio.
3. **Robust Optimisation:** Minimise risk under the *worst case* within an uncertainty set around parameters.
4. **Minimum Variance / Risk Parity:** Avoid return forecasts entirely.

**Source:** Michaud, R. (1989). *The Markowitz Optimisation Enigma*. Financial Analysts Journal; Black & Litterman (1992)

In [ ]:
np.random.seed(0)
n_assets = 5
names5   = ['SAP', 'Allianz', 'Siemens', 'BASF', 'RWE']
true_mu  = np.array([0.10, 0.08, 0.09, 0.07, 0.06])
vols5    = np.array([0.22, 0.18, 0.20, 0.24, 0.16])
corr5    = np.array([[1,.4,.5,.4,.2],[.4,1,.5,.4,.3],[.5,.5,1,.5,.3],
                     [.4,.4,.5,1,.3],[.2,.3,.3,.3,1]])
cov5     = np.outer(vols5, vols5) * corr5

def max_sharpe_weights(mu, cov, rf=RISK_FREE):
    n = len(mu); w0 = np.ones(n)/n
    res = minimize(lambda w: -(w@mu - rf)/np.sqrt(w@cov@w), w0,
                   method='SLSQP', bounds=[(0,1)]*n,
                   constraints=[{'type':'eq','fun':lambda w: w.sum()-1}],
                   options={'ftol':1e-12,'maxiter':1000})
    return res.x

w_orig  = max_sharpe_weights(true_mu, cov5)
w_boots = []
for _ in range(50):
    mu_noisy = true_mu + np.random.normal(0, 0.005, n_assets)
    try:
        w_boots.append(max_sharpe_weights(mu_noisy, cov5))
    except:
        pass
w_boots = np.array(w_boots)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bp = axes[0].boxplot(w_boots * 100, labels=names5, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#4a90d9'); patch.set_alpha(0.6)
axes[0].plot(range(1, n_assets+1), w_orig * 100, 'r*', ms=12, zorder=5, label='True Optimum')
axes[0].set_ylabel('Portfolio Weight (%)')
axes[0].set_title('Estimation Error Problem:\nWeight Instability with ±0.5% Return Estimation Error',
                  fontweight='bold')
axes[0].legend()

weight_stds = w_boots.std(axis=0) * 100
axes[1].bar(names5, weight_stds, color='#e74c3c', edgecolor='white')
axes[1].set_ylabel('Std. Dev. of Weights (%)')
axes[1].set_title('Optimisation Instability\n(High = Unstable Weight)', fontweight='bold')

plt.tight_layout()
plt.savefig('estimation_error.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key finding: Small estimation errors lead to dramatically different optimal portfolios!')
print('→ This is why Minimum Variance and Risk Parity are often preferred in practice.')

---
## Summary – Module 2

| Concept | Core Formula | Key Insight |
|---------|-------------|-------------|
| **Sharpe Ratio** | $(\mu_p - r_f)/\sigma_p$ | Penalises up- and downside volatility equally |
| **Sortino Ratio** | $(\mu_p - r_f)/\sigma_{\text{downside}}$ | Only downside penalised; better for skewed strategies |
| **Information Ratio** | $\alpha / TE$ | Measures manager quality; Fundamental Law: IR ≈ IC·√BR |
| **CAPM** | $E[r_i] = r_f + \beta(r_m - r_f)$ | Only market risk is rewarded; SML |
| **VaR** | $-Q_{1-\alpha}(P\&L)$ | Not coherent; Basel II standard |
| **CVaR/ES** | $-E[P\&L \mid P\&L \leq -VaR]$ | Coherent; Basel III standard (97.5%) |
| **Estimation Error** | — | Markowitz amplifies forecast errors → robust alternatives needed |
| **Black-Litterman** | Prior + Views → Posterior | Solves estimation error via market equilibrium as prior |

---
**→ Continue with Module 3: Strategies, Backtesting & Instruments**